# G0 — Batch Scaling Sweep

**Model**: Qwen2.5-1.5B-Instruct  
**Workload**: 50 req × 128/64 tokens, capacity + cpu-first  
**num_seqs sweep**: 1, 2, 4, 8, 16  
**측정 모드**: `VLLM_HYBRID_PROFILE=1 VLLM_HYBRID_PROFILE_SUBLAYER=1`

각 run 의 `hybrid_server_run.log` 에서 `[HYBRID-CPU-PROFILE]` sublayer breakdown 과 bench 의 wall / tok/s 를 집계.

In [ ]:
import os, re, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# sweep dir 에 notebook 복사 후 열면 자동 감지 (Path.cwd).
# 또는 G0_ROOT env 로 명시: G0_ROOT=/path/to/g0_NN jupyter notebook ...
ROOT = Path(os.environ.get('G0_ROOT', Path.cwd()))
print(f'ROOT = {ROOT}  (exists: {ROOT.exists()})')
SEQS = [1, 2, 4, 8, 16]
print('run dirs:', [(s, (ROOT / f'seqs{s}').exists()) for s in SEQS])

## 1. Profile 로그 파싱 — sublayer breakdown 시계열

In [ ]:
# [HYBRID-CPU-PROFILE] sublayer line regex
# 예: step=5 reqs=0 tokens=1 threads=16 total=93.4ms qkv=4.8ms(28) o=3.8ms(28) gate_up=38.4ms(28) down=19.4ms(28) norm=2.9ms(56) act=0.8ms(28) attn_core=17.9ms(28) other=16.6ms
LINE_RE = re.compile(
    r'\[HYBRID-CPU-PROFILE\] step=(\d+) reqs=(\S+) tokens=(\d+) '
    r'threads=(\d+) total=([\d.]+)ms '
    r'qkv=([\d.]+)ms\((\d+)\) o=([\d.]+)ms\((\d+)\) '
    r'gate_up=([\d.]+)ms\((\d+)\) down=([\d.]+)ms\((\d+)\) '
    r'norm=([\d.]+)ms\((\d+)\) act=([\d.]+)ms\((\d+)\) '
    r'attn_core=([\d.]+)ms\((\d+)\) other=([\d.]+)ms'
)

def parse_profile(log_path):
    rows = []
    if not log_path.exists():
        return pd.DataFrame()
    for line in log_path.read_text(errors='replace').splitlines():
        m = LINE_RE.search(line)
        if not m:
            continue
        g = m.groups()
        rows.append({
            'step': int(g[0]), 'reqs': g[1], 'tokens': int(g[2]),
            'threads': int(g[3]), 'total_ms': float(g[4]),
            'qkv_ms': float(g[5]), 'qkv_n': int(g[6]),
            'o_ms': float(g[7]), 'o_n': int(g[8]),
            'gate_up_ms': float(g[9]), 'gate_up_n': int(g[10]),
            'down_ms': float(g[11]), 'down_n': int(g[12]),
            'norm_ms': float(g[13]), 'norm_n': int(g[14]),
            'act_ms': float(g[15]), 'act_n': int(g[16]),
            'attn_core_ms': float(g[17]), 'attn_core_n': int(g[18]),
            'other_ms': float(g[19]),
        })
    return pd.DataFrame(rows)

dfs = {}
for s in SEQS:
    p = ROOT / f'seqs{s}' / 'hybrid_server_run.log'
    dfs[s] = parse_profile(p)
    print(f'seqs={s}: {len(dfs[s])} profile samples  (log={p.exists()})')

## 2. Bench 결과 (wall, TPOT, TTFT)

In [ ]:
def parse_bench(bench_dir):
    j = bench_dir / 'hybrid.json'
    if not j.exists():
        return {}
    return json.loads(j.read_text())

bench_rows = []
for s in SEQS:
    b = parse_bench(ROOT / f'seqs{s}')
    if not b:
        continue
    bench_rows.append({
        'seqs': s,
        'wall_s': b.get('duration', None),
        'req_throughput': b.get('request_throughput', None),
        'out_tps': b.get('output_throughput', None),
        'tpot_mean': b.get('mean_tpot_ms', None),
        'tpot_p99': b.get('p99_tpot_ms', None),
        'ttft_mean': b.get('mean_ttft_ms', None),
        'ttft_p99': b.get('p99_ttft_ms', None),
    })
bench_df = pd.DataFrame(bench_rows)
bench_df

## 3. Batch scaling — step time per num_seqs

In [ ]:
# 각 run 의 profile steps 에서 median total_ms + sublayer 기여도
summary = []
for s in SEQS:
    df = dfs[s]
    if df.empty:
        continue
    # 초기 몇 step 은 warmup 이므로 drop
    d = df.iloc[3:] if len(df) > 5 else df
    summary.append({
        'seqs': s,
        'samples': len(d),
        'total_med': d['total_ms'].median(),
        'qkv_med': d['qkv_ms'].median(),
        'o_med': d['o_ms'].median(),
        'gate_up_med': d['gate_up_ms'].median(),
        'down_med': d['down_ms'].median(),
        'norm_med': d['norm_ms'].median(),
        'act_med': d['act_ms'].median(),
        'attn_core_med': d['attn_core_ms'].median(),
        'other_med': d['other_ms'].median(),
    })
if not summary:
    print('⚠ No [HYBRID-CPU-PROFILE] markers found in any hybrid_server_run.log')
    print('  → sublayer analysis skipped. Bench (wall/TPOT) analysis still works.')
    sdf = pd.DataFrame(columns=['seqs','samples','total_med']).set_index('seqs')
else:
    sdf = pd.DataFrame(summary).set_index('seqs')
# batch_scaling_ratio = total(N) / total(1)
base = sdf.loc[1, 'total_med'] if 1 in sdf.index else sdf.iloc[0]['total_med']
sdf['scaling_ratio'] = sdf['total_med'] / base
sdf['linear_expected'] = sdf.index.values / sdf.index[0]
sdf['scaling_efficiency'] = sdf['linear_expected'] / sdf['scaling_ratio']
sdf

In [ ]:
# Plot 1: sublayer stacked bars (per num_seqs)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
cols = ['qkv_med', 'o_med', 'gate_up_med', 'down_med', 'norm_med',
        'act_med', 'attn_core_med', 'other_med']
labels = ['QKV', 'O proj', 'Gate+Up', 'Down', 'Norm', 'Act (SiLU)',
          'Attn core', 'Other']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
          '#8c564b', '#e377c2', '#7f7f7f']

bottom = np.zeros(len(sdf))
for c, label, color in zip(cols, labels, colors):
    vals = sdf[c].values
    ax.bar(sdf.index.astype(str), vals, bottom=bottom, label=label,
           color=color, edgecolor='white', linewidth=0.5)
    bottom += vals
ax.set_xlabel('num_seqs (cpu_max_num_seqs)')
ax.set_ylabel('Step time (ms, median)')
ax.set_title('Sublayer breakdown per num_seqs')
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(True, axis='y', alpha=0.3)

# Plot 2: scaling ratio vs linear expectation
ax = axes[1]
seqs_arr = sdf.index.astype(float).values
ax.plot(seqs_arr, sdf['scaling_ratio'].values, 'o-',
        color='#d62728', linewidth=2, markersize=10,
        label='Measured scaling ratio')
ax.plot(seqs_arr, seqs_arr / seqs_arr[0], '--',
        color='gray', linewidth=1.5, label='Linear expectation')
ax.plot(seqs_arr, np.ones_like(seqs_arr), ':',
        color='green', linewidth=1.5, label='Perfect amortization')
ax.set_xscale('log', base=2)
ax.set_xticks(seqs_arr)
ax.set_xticklabels([str(int(s)) for s in seqs_arr])
ax.set_xlabel('num_seqs')
ax.set_ylabel('step(N) / step(1)')
ax.set_title('Batch scaling — dev CPU engine')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ROOT / 'analysis_summary.png',
            dpi=130, bbox_inches='tight')
plt.show()

## 4. Per-sublayer scaling (어느 sublayer 가 폭증?)

In [ ]:
# 각 sublayer 의 step(N) / step(1) 비율
sub_cols = ['qkv_med', 'o_med', 'gate_up_med', 'down_med',
            'norm_med', 'act_med', 'attn_core_med']
sub_labels = ['QKV', 'O proj', 'Gate+Up', 'Down', 'Norm', 'Act', 'Attn core']

if 1 in sdf.index:
    base_row = sdf.loc[1]
    ratio_df = pd.DataFrame({
        label: sdf[col] / base_row[col]
        for col, label in zip(sub_cols, sub_labels)
    })
    ratio_df['linear'] = sdf.index / 1
else:
    ratio_df = None
ratio_df

In [ ]:
if ratio_df is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    xs = ratio_df.index.astype(float).values
    for label, color in zip(sub_labels, colors[:len(sub_labels)]):
        ax.plot(xs, ratio_df[label].values, 'o-', label=label, color=color)
    ax.plot(xs, ratio_df['linear'].values, '--', color='gray',
            linewidth=1.5, label='Linear (N/1)')
    ax.set_xscale('log', base=2)
    ax.set_xticks(xs)
    ax.set_xticklabels([str(int(s)) for s in xs])
    ax.set_xlabel('num_seqs')
    ax.set_ylabel('sublayer step(N) / step(1)')
    ax.set_title('Per-sublayer batch scaling — top bottleneck 식별')
    ax.legend(fontsize=9, ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Bench wall vs num_seqs

In [ ]:
if not bench_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.plot(bench_df['seqs'], bench_df['wall_s'], 'o-', color='#d62728',
            markersize=10, linewidth=2)
    ax.set_xscale('log', base=2)
    ax.set_xticks(bench_df['seqs'])
    ax.set_xticklabels([str(int(s)) for s in bench_df['seqs']])
    ax.set_xlabel('num_seqs')
    ax.set_ylabel('wall (s)')
    ax.set_title('Bench wall vs num_seqs (50 req x 128/64)')
    ax.grid(True, alpha=0.3)
    for _, r in bench_df.iterrows():
        ax.annotate(f"{r['wall_s']:.1f}s", (r['seqs'], r['wall_s']),
                    textcoords='offset points', xytext=(10, 5))

    ax = axes[1]
    ax.plot(bench_df['seqs'], bench_df['tpot_mean'], 'o-',
            label='TPOT mean', color='#1f77b4', markersize=10)
    ax.plot(bench_df['seqs'], bench_df['tpot_p99'], 's--',
            label='TPOT p99', color='#ff7f0e', markersize=8)
    ax.set_xscale('log', base=2)
    ax.set_xticks(bench_df['seqs'])
    ax.set_xticklabels([str(int(s)) for s in bench_df['seqs']])
    ax.set_xlabel('num_seqs')
    ax.set_ylabel('TPOT (ms)')
    ax.set_title('TPOT vs num_seqs')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Applied Features Manifest 확인

In [ ]:
for s in SEQS:
    mpath = ROOT / f'seqs{s}' / 'applied_features.json'
    if not mpath.exists():
        print(f'seqs={s}: no manifest')
        continue
    m = json.loads(mpath.read_text())
    enabled = {k: v for k, v in m['features'].items()
               if v and v != '0' and v != 'off'}
    print(f"seqs={s}: git={m['git_sha'][:8]} "
          f"profile={m['profile_meta']['VLLM_HYBRID_PROFILE']} "
          f"sublayer={m['profile_meta']['VLLM_HYBRID_PROFILE_SUBLAYER']} "
          f"enabled={list(enabled.keys()) or 'none'}")

## 7. G0 Gate 판정

- **G0 통과 조건**: num_seqs sweep 확보 + sublayer breakdown
- **G1 조건**: `scaling_ratio(4/1) ≤ 2×`
- **G2 조건**: `scaling_ratio(4/1) ≤ 1.5×`

In [ ]:
# Top bottleneck identification
if 1 in sdf.index:
    top = sdf.loc[1, sub_cols].sort_values(ascending=False)
    print('Top bottleneck @ seqs=1:')
    for c, v in top.items():
        label = sub_labels[sub_cols.index(c)]
        pct = v / sdf.loc[1, 'total_med'] * 100
        print(f'  {label:12s} {v:6.2f}ms ({pct:.1f}%)')

# Gate evaluation
if 4 in sdf.index and 1 in sdf.index:
    ratio_4_1 = sdf.loc[4, 'total_med'] / sdf.loc[1, 'total_med']
    print(f'\nscaling_ratio(4/1) = {ratio_4_1:.2f}x')
    if ratio_4_1 <= 1.5:
        print('  -> G2 pass')
    elif ratio_4_1 <= 2.0:
        print('  -> G1 pass')
    else:
        print('  -> pre-G1 (batch scaling 실패)')

if 16 in sdf.index and 1 in sdf.index:
    ratio_16_1 = sdf.loc[16, 'total_med'] / sdf.loc[1, 'total_med']
    print(f'scaling_ratio(16/1) = {ratio_16_1:.2f}x (linear 기대 16x)')
    print(f'  failure factor = {16/ratio_16_1:.1f}x (16/ratio)')

## 8. CPU / GPU Utilization (monitor CSV 기반)

In [ ]:
# Monitor CSV 로드 (per seqs)
monitors = {}
for s in SEQS:
    cpu_p = ROOT / f'seqs{s}' / 'hybrid_monitor_cpu.csv'
    gpu_p = ROOT / f'seqs{s}' / 'hybrid_monitor_gpu.csv'
    if cpu_p.exists() and gpu_p.exists():
        monitors[s] = {'cpu': pd.read_csv(cpu_p), 'gpu': pd.read_csv(gpu_p)}
print({s: (len(m['cpu']), len(m['gpu'])) for s, m in monitors.items()})

### 8.1 CPU / GPU 평균 utilization 시계열 (per seqs)

In [ ]:
if monitors:
    n = len(monitors)
    fig, axes = plt.subplots(n, 1, figsize=(14, 2.5*n), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, (s, m) in zip(axes, sorted(monitors.items())):
        ax.plot(m['cpu']['elapsed_s'], m['cpu']['cpu_avg_util_pct'],
                label='CPU avg %', color='#1f77b4', linewidth=1.5)
        ax.plot(m['gpu']['elapsed_s'], m['gpu']['gpu_avg_util_pct'],
                label='GPU avg %', color='#d62728', linewidth=1.5)
        ax.set_ylabel(f'seqs={s}\nutil %')
        ax.set_ylim(0, 105)
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel('elapsed (s)')
    plt.suptitle('CPU / GPU avg utilization — per seqs', y=1.001)
    plt.tight_layout()
    plt.savefig(ROOT / 'analysis_util_timeseries.png', dpi=130, bbox_inches='tight')
    plt.show()
else:
    print('no monitor CSVs found')

### 8.2 GPU 전력 / 메모리 시계열

H100x8 서버 레벨 전력과 메모리 사용량을 bench 진행 동안 추적.

In [ ]:
if monitors:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for s, m in sorted(monitors.items()):
        axes[0].plot(m['gpu']['elapsed_s'], m['gpu']['gpu_avg_power_w'], label=f'seqs={s}')
        axes[1].plot(m['gpu']['elapsed_s'], m['gpu']['gpu_avg_mem_util_pct'], label=f'seqs={s}')
    axes[0].set_xlabel('elapsed (s)'); axes[0].set_ylabel('GPU avg power (W)')
    axes[0].set_title('GPU power draw'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    axes[1].set_xlabel('elapsed (s)'); axes[1].set_ylabel('GPU avg mem util (%)')
    axes[1].set_title('GPU memory util'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(ROOT / 'analysis_gpu_power_mem.png', dpi=130, bbox_inches='tight')
    plt.show()

### 8.3 Per-CPU utilization heatmap (가장 큰 seqs)

X축 = 시간(샘플), Y축 = CPU core #, 색 = util %.  CPU affinity 및 NUMA 활용 패턴 시각화.

In [ ]:
if monitors:
    s = max(monitors.keys())
    df = monitors[s]['cpu']
    cpu_cols = [c for c in df.columns
                if c.startswith('cpu') and c.endswith('_util_pct')
                and c != 'cpu_avg_util_pct']
    mat = df[cpu_cols].T.values  # (n_cpus, n_samples)
    fig, ax = plt.subplots(figsize=(14, max(4, len(cpu_cols) * 0.03)))
    im = ax.imshow(mat, aspect='auto', cmap='hot',
                   interpolation='nearest', vmin=0, vmax=100,
                   extent=[df['elapsed_s'].iloc[0], df['elapsed_s'].iloc[-1],
                           len(cpu_cols), 0])
    ax.set_xlabel('elapsed (s)')
    ax.set_ylabel('CPU core #')
    ax.set_title(f'Per-CPU utilization heatmap — seqs={s} (n_cpus={len(cpu_cols)})')
    plt.colorbar(im, ax=ax, label='util %')
    plt.tight_layout()
    plt.savefig(ROOT / 'analysis_cpu_heatmap.png', dpi=130, bbox_inches='tight')
    plt.show()

### 8.4 Util summary (per seqs)

In [ ]:
if monitors:
    rows = []
    for s, m in sorted(monitors.items()):
        rows.append({
            'seqs': s,
            'cpu_mean_%': m['cpu']['cpu_avg_util_pct'].mean(),
            'cpu_max_%': m['cpu']['cpu_avg_util_pct'].max(),
            'gpu_mean_%': m['gpu']['gpu_avg_util_pct'].mean(),
            'gpu_max_%': m['gpu']['gpu_avg_util_pct'].max(),
            'gpu_mem_mean_%': m['gpu']['gpu_avg_mem_util_pct'].mean(),
            'gpu_power_mean_W': m['gpu']['gpu_avg_power_w'].mean(),
        })
    udf = pd.DataFrame(rows).set_index('seqs').round(2)
    udf